In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
data = pd.read_csv('../data/raw/TS_price_history.csv')
app_list = pd.read_csv('../data/raw/steam_app_list.csv')

In [5]:
appid_map = app_list.set_index('Title')['ID'].to_dict()

case_insensitive_appid_map = {
    str(title).casefold(): app_id
    for title, app_id in appid_map.items()
    if not (isinstance(title, float) and np.isnan(title))
}

data['app_id'] = data['app_id'].fillna(
    data['game_title'].astype(str).str.casefold().map(case_insensitive_appid_map)
)

In [6]:
missing_rows = data[data.isna().any(axis=1)]
print(f"Missing rows: {len(missing_rows)}")
missing_rows

Missing rows: 2794


,app_id,game_title,date,price,regular_price,sales_percentage,shop_id,shop_name,currency,is_historical_low
115852,NaN,Grand Theft Auto V,6/24/2014,59.99,59.99,0,36,GreenManGaming,USD,FALSE
115853,NaN,Grand Theft Auto V,9/24/2014,61.95,61.95,0,31,Gamesrocket,USD,FALSE
115854,NaN,Grand Theft Auto V,11/8/2014,59.95,59.95,0,31,Gamesrocket,USD,FALSE
115855,NaN,Grand Theft Auto V,1/3/2015,57.95,57.95,0,31,Gamesrocket,USD,FALSE
115856,NaN,Grand Theft Auto V,1/16/2015,48.44,60.55,20,26,GamesPlanet UK,USD,FALSE
...,...,...,...,...,...,...,...,...,...,...
203020,NaN,Rocket League,3/9/2020,9.99,19.99,50,61,Steam,USD,FALSE
203021,NaN,Rocket League,3/16/2020,19.99,19.99,0,61,Steam,USD,FALSE
203022,NaN,Rocket League,3/25/2020,9.99,19.99,50,61,Steam,USD,FALSE
203023,NaN,Rocket League,4/20/2020,19.99,19.99,0,61,Steam,USD,FALSE


In [7]:
duplicate_rows = data[data.duplicated(keep=False)]
duplicate_rows

,app_id,game_title,date,price,regular_price,sales_percentage,shop_id,shop_name,currency,is_historical_low
170,752590.0,A Plague Tale: Innocence,1/22/2020,22.49,44.99,50,37,Humble Store,USD,FALSE
171,752590.0,A Plague Tale: Innocence,1/22/2020,44.99,44.99,0,37,Humble Store,USD,FALSE
172,752590.0,A Plague Tale: Innocence,1/22/2020,22.49,44.99,50,37,Humble Store,USD,FALSE
173,752590.0,A Plague Tale: Innocence,1/22/2020,44.99,44.99,0,37,Humble Store,USD,FALSE
1806,752590.0,A Plague Tale: Innocence,8/1/2024,34.39,39.99,14,6,Fanatical,USD,FALSE
...,...,...,...,...,...,...,...,...,...,...
267589,200510.0,XCOM: Enemy Unknown,7/29/2024,29.99,29.99,0,67,eTail.Market,USD,FALSE
269580,3240220.0,Grand Theft Auto V Enhanced,11/30/2023,29.98,29.98,0,24,GamersGate,USD,FALSE
269582,3240220.0,Grand Theft Auto V Enhanced,11/30/2023,29.98,29.98,0,24,GamersGate,USD,FALSE
269789,3240220.0,Grand Theft Auto V Enhanced,8/1/2024,26.75,29.98,11,6,Fanatical,USD,FALSE


In [8]:
# game_info = pd.read_csv('gfcheckpoint.csv')

In [9]:
# data.drop(columns=['game_key'], inplace=True)

In [10]:
data.tail(5)

,app_id,game_title,date,price,regular_price,sales_percentage,shop_id,shop_name,currency,is_historical_low
270380,3240220.0,Grand Theft Auto V Enhanced,5/7/2026,13.20,29.99,56,36,GreenManGaming,USD,FALSE
270381,3240220.0,Grand Theft Auto V Enhanced,5/7/2026,14.99,29.99,50,37,Humble Store,USD,FALSE
270382,3240220.0,Grand Theft Auto V Enhanced,5/8/2026,12.90,29.99,57,36,GreenManGaming,USD,FALSE
270383,3240220.0,Grand Theft Auto V Enhanced,5/11/2026,13.20,29.99,56,36,GreenManGaming,USD,FALSE
270384,3240220.0,Grand Theft Auto V Enhanced,5/12/2026,29.99,29.99,0,36,GreenManGaming,USD,FALSE


In [11]:
data['date'] = pd.to_datetime(data['date'])

In [12]:
print("=== SORTING DATA ===\n")
print(f"Original shape: {data.shape}")

# Sort by app_id first (maintain grouping), then by shop, then by date (chronologically)
data = data.sort_values(['app_id', 'shop_name', 'date']).reset_index(drop=True)

print(f"Data sorted by: app_id → shop_name → date")
print(f"Final shape: {data.shape}")
print(f"\nSample of sorted data:")
print(data[['app_id', 'shop_name', 'game_title', 'date', 'price']].head(15))

=== SORTING DATA ===

Original shape: (270385, 10)
Data sorted by: app_id → shop_name → date
Final shape: (270385, 10)

Sample of sorted data:
    app_id       shop_name              game_title       date  price
0    240.0         GameFly  Counter-Strike: Source 2013-05-07   4.99
1    240.0         GameFly  Counter-Strike: Source 2013-07-25  19.99
2    240.0         GameFly  Counter-Strike: Source 2013-08-22   7.99
3    240.0         GameFly  Counter-Strike: Source 2013-08-27  19.99
4    240.0         GameFly  Counter-Strike: Source 2013-12-14   4.99
5    240.0         GameFly  Counter-Strike: Source 2013-12-16  19.99
6    240.0         GameFly  Counter-Strike: Source 2014-03-28   4.99
7    240.0         GameFly  Counter-Strike: Source 2014-04-01  19.99
8    240.0  GreenManGaming  Counter-Strike: Source 2013-11-29   9.99
9    240.0  GreenManGaming  Counter-Strike: Source 2013-12-03  19.99
10   240.0  GreenManGaming  Counter-Strike: Source 2013-12-20   4.99
11   240.0  GreenManGaming  C

In [13]:
print("=== CLEANING STEPS ===\n")
print(f"Initial shape: {data.shape}")

# Remove missing app_id
data_clean = data.dropna(subset=['app_id']).copy()
print(f"After removing missing app_id: {data_clean.shape}")

# Convert date to datetime
data_clean['date'] = pd.to_datetime(data_clean['date'])

data_clean['app_id'] = data_clean['app_id'].astype(str).str.rstrip('0').str.rstrip('.')

# Remove exact duplicates
initial_shape = data_clean.shape[0]
data_clean = data_clean.drop_duplicates()
print(f"Exact duplicates removed: {initial_shape - data_clean.shape[0]} rows")
print(f"After removing exact duplicates: {data_clean.shape}")

# Remove records with same app_id, date, and shop_name (keep only latest)
before_dedup = data_clean.shape[0]
data_clean = data_clean.drop_duplicates(subset=['app_id', 'date', 'shop_name'], keep='last')
print(f"Same-day price changes removed: {before_dedup - data_clean.shape[0]} rows (kept latest)")
print(f"After removing same-day duplicates: {data_clean.shape}")

# Remove rows with invalid prices (negative prices)
data_clean = data_clean[data_clean['price'] >= 0]
data_clean = data_clean[data_clean['regular_price'] >= 0]
print(f"After removing invalid prices: {data_clean.shape}")

# Remove invalid discount scenarios (regular_price < price)
before_discount_check = data_clean.shape[0]
data_clean = data_clean[data_clean['regular_price'] >= data_clean['price']]
print(f"Invalid discount records removed: {before_discount_check - data_clean.shape[0]} rows")
print(f"After removing invalid discounts: {data_clean.shape}")

# Filter prices to reasonable range (0-80 based on EDA)
before_range = data_clean.shape[0]
data_clean = data_clean[(data_clean['price'] <= 80) & (data_clean['regular_price'] <= 80)]
print(f"Price range filter (0-80) removed: {before_range - data_clean.shape[0]} rows")
print(f"After filtering price range: {data_clean.shape}")

# Remove extreme outliers (prices over 10000) - additional safety check
data_clean = data_clean[data_clean['price'] <= 10000]
data_clean = data_clean[data_clean['regular_price'] <= 10000]
print(f"After removing extreme outliers: {data_clean.shape}")

# Remove rows where sales_percentage is unreasonable (< -100 or > 100)
data_clean = data_clean[(data_clean['sales_percentage'] >= -100) & (data_clean['sales_percentage'] <= 100)]
print(f"After removing invalid sales percentages: {data_clean.shape}")

# Remove rows where currency is not USD
data_clean = data_clean[data_clean['currency'] == 'USD']
print(f"After filtering for USD currency: {data_clean.shape}")

print(f"\n=== FINAL RESULT ===")
print(f"Final shape: {data_clean.shape}")
print(f"Total rows removed: {initial_shape - data_clean.shape[0]}")
print(f"\nData types:\n{data_clean.dtypes}")
print(f"\nMissing values:\n{data_clean.isnull().sum()}")

=== CLEANING STEPS ===

Initial shape: (270385, 10)
After removing missing app_id: (267591, 10)
Exact duplicates removed: 11867 rows
After removing exact duplicates: (255724, 10)
Same-day price changes removed: 22719 rows (kept latest)
After removing same-day duplicates: (233005, 10)
After removing invalid prices: (233005, 10)
Invalid discount records removed: 646 rows
After removing invalid discounts: (232359, 10)
Price range filter (0-80) removed: 163 rows
After filtering price range: (232196, 10)
After removing extreme outliers: (232196, 10)
After removing invalid sales percentages: (232196, 10)
After filtering for USD currency: (222449, 10)

=== FINAL RESULT ===
Final shape: (222449, 10)
Total rows removed: 45142

Data types:
app_id                        object
game_title                    object
date                  datetime64[ns]
price                        float64
regular_price                float64
sales_percentage               int64
shop_id                        int64
s

In [14]:
# # Remove games released after 2023 or with fewer than 90 records in data_clean
# game_info_filtered = game_info.copy()
# game_info_filtered['release_date'] = pd.to_datetime(game_info_filtered['release_date'], errors='coerce')

# title_counts = data_clean['game_title'].value_counts(dropna=False)

# valid_titles = (
#     game_info_filtered.loc[
#         game_info_filtered['release_date'].isna() | (game_info_filtered['release_date'].dt.year <= 2023),
#         'game_title'
#     ]
#     .dropna()
#     .astype(str)
# )

# valid_titles = set(valid_titles[valid_titles.map(title_counts).fillna(0) >= 90])

# removed_game_info = len(game_info) - game_info['game_title'].astype(str).isin(valid_titles).sum()
# removed_data_clean = len(data_clean) - data_clean['game_title'].astype(str).isin(valid_titles).sum()

# game_info = game_info[game_info['game_title'].astype(str).isin(valid_titles)].copy()
# data_clean = data_clean[data_clean['game_title'].astype(str).isin(valid_titles)].copy()

# print(f"Removed from game_info: {removed_game_info}")
# print(f"Removed from data_clean: {removed_data_clean}")
# print(f"Remaining game_info rows: {len(game_info)}")
# print(f"Remaining data_clean rows: {len(data_clean)}")

In [15]:
print("=== CLEANED DATA VERIFICATION ===\n")
print(f"Sample of cleaned data:")
print(data_clean.head(10))
print(f"\nData summary:")
print(data_clean.describe())

=== CLEANED DATA VERIFICATION ===

Sample of cleaned data:
  app_id              game_title       date  price  regular_price  \
0    240  Counter-Strike: Source 2013-05-07   4.99          19.99   
1    240  Counter-Strike: Source 2013-07-25  19.99          19.99   
2    240  Counter-Strike: Source 2013-08-22   7.99          19.99   
3    240  Counter-Strike: Source 2013-08-27  19.99          19.99   
4    240  Counter-Strike: Source 2013-12-14   4.99          19.99   
5    240  Counter-Strike: Source 2013-12-16  19.99          19.99   
6    240  Counter-Strike: Source 2014-03-28   4.99          19.99   
7    240  Counter-Strike: Source 2014-04-01  19.99          19.99   
8    240  Counter-Strike: Source 2013-11-29   9.99          19.99   
9    240  Counter-Strike: Source 2013-12-03  19.99          19.99   

   sales_percentage  shop_id       shop_name currency is_historical_low   
0                75       21         GameFly      USD             FALSE   
1                 0       21   

In [16]:
print("=== DATA QUALITY CHECKS ===\n")

# Check 1: Price range analysis
print(f"Price range: {data_clean['price'].min():.2f} - {data_clean['price'].max():.2f}")
print(f"Regular price range: {data_clean['regular_price'].min():.2f} - {data_clean['regular_price'].max():.2f}")
prices_above_80 = len(data_clean[data_clean['price'] > 80])
regular_above_80 = len(data_clean[data_clean['regular_price'] > 80])
print(f"Records with price > 80: {prices_above_80}")
print(f"Records with regular_price > 80: {regular_above_80}")

# Check 2: Invalid discount scenarios (regular_price < price)
invalid_discount = data_clean[data_clean['regular_price'] < data_clean['price']]
print(f"\nRecords where regular_price < price (invalid discounts): {len(invalid_discount)}")
if len(invalid_discount) > 0:
    print(f"Sample: {invalid_discount[['game_title', 'price', 'regular_price', 'sales_percentage']].head()}")

# Check 3: Sales percentage vs actual calculation
data_clean['calculated_discount'] = ((data_clean['regular_price'] - data_clean['price']) / data_clean['regular_price'] * 100).round(0)
data_clean['calculated_discount'] = data_clean['calculated_discount'].fillna(0).astype(int)
discount_mismatch = data_clean[data_clean['sales_percentage'] != data_clean['calculated_discount']]
print(f"\nRecords where sales_percentage doesn't match calculation: {len(discount_mismatch)}")
if len(discount_mismatch) > 0:
    print(f"Sample mismatch: {discount_mismatch[['price', 'regular_price', 'sales_percentage', 'calculated_discount']].head()}")
    # Replace sales_percentage with calculated_discount where applicable
    data_clean['sales_percentage'] = data_clean['calculated_discount'].fillna(data_clean['sales_percentage']).astype(int)

    # Drop the calculated_discount column as it's no longer needed
    data_clean = data_clean.drop(columns=['calculated_discount'])

    print(f"\nSales percentage updated based on calculated discounts")
    print(f"Final data_clean shape: {data_clean.shape}")
# Check 4: app_id validation (should be numeric)
try:
    app_id_numeric = pd.to_numeric(data_clean['app_id'], errors='coerce')
    invalid_app_id = data_clean[app_id_numeric.isna()]
    print(f"\nRecords with non-numeric app_id: {len(invalid_app_id)}")
    if len(invalid_app_id) > 0:
        print(f"Sample: {invalid_app_id['app_id'].unique()[:5]}")
except:
    print("\nCould not convert app_id to numeric")

# Check 5: Date range
print(f"\nDate range: {data_clean['date'].min()} to {data_clean['date'].max()}")

# Check 6: Missing values in game_title
print(f"\nMissing game_title: {data_clean['game_title'].isna().sum()}")
print(f"Unique games: {data_clean['game_title'].nunique()}")
print(f"Unique shops: {data_clean['shop_name'].nunique()}")


=== DATA QUALITY CHECKS ===

Price range: 0.00 - 79.99
Regular price range: 0.00 - 79.99
Records with price > 80: 0
Records with regular_price > 80: 0

Records where regular_price < price (invalid discounts): 0

Records where sales_percentage doesn't match calculation: 609
Sample mismatch:       price  regular_price  sales_percentage  calculated_discount
1378  26.88          29.99                13                   10
1393  26.88          29.99                11                   10
1975   5.48          29.99                81                   82
2465   5.47          29.99                81                   82
2468   6.44          29.99                78                   79

Sales percentage updated based on calculated discounts
Final data_clean shape: (222449, 10)

Records with non-numeric app_id: 0

Date range: 2012-02-10 00:00:00 to 2026-05-12 00:00:00

Missing game_title: 0
Unique games: 216
Unique shops: 59


In [17]:
from pathlib import Path
output_dir = 'data/processed'
# save to data/processed located one level up from the notebooks folder
target_dir = Path.cwd().parent / Path(output_dir)
target_dir.mkdir(parents=True, exist_ok=True)

output_path = target_dir / 'TS_history_cleaned.csv'
data_clean.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Saved to d:\NEU\Time-series Analysis\Steam-Sale-Predict\data\processed\TS_history_cleaned.csv
